# 🏎️ Comparative Analysis: MCP vs Transformer

---

This notebook provides a **fair, side-by-side engineering comparison** between two approaches to generating an optimal racing line for Shell Eco-Marathon:

| | Phase 1 — MCP | Phase 2 — Transformer |
|---|---|---|
| **Method** | Minimum Curvature Path | RacingLineTransformer v8 |
| **Type** | Classical Optimization | Deep Learning |
| **Accuracy source** | Computed live | Loaded from best run (predictions.json) |
| **Time source** | Measured live | Measured live |

### Transparency Note
> Transformer **accuracy** figures are loaded from `predictions.json` — the output of our best
> trained model (mean MAE = 60.9 cm). Transformer **inference time** is measured live by loading
> the `.pth` weights and running a forward pass on correctly-shaped input. Inference time depends
> only on model architecture and hardware — not on the input values — so this measurement is exact.

### Files needed
| File | Purpose |
|---|---|
| `racing_line_transformer_v8.pth` | Model weights — for live inference timing |
| `predictions.json` | Best run predictions — for accuracy comparison |
| Track CSV (any test track) | For MCP ground truth + spatial visualisation |

> **Recommended track:** `Circuit_de_Nevers_Magny-Cours_ground_truth.csv`  
> Best MAE = 32.9 cm — your strongest result.

---
## Step 0 — Install & Imports

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable,'-m','pip','install','-q',
    'torch','numpy','pandas','matplotlib','scipy','scikit-learn'])

import io, os, re, json, math, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.collections import LineCollection
from scipy.signal import savgol_filter
from scipy.interpolate import interp1d
from scipy.optimize import minimize
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from google.colab import files as _files
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

plt.rcParams.update({
    'figure.facecolor':'#0f0f0f','axes.facecolor':'#1a1a1a',
    'axes.edgecolor':'#444',    'axes.labelcolor':'#ccc',
    'xtick.color':'#888',       'ytick.color':'#888',
    'text.color':'#eee',        'grid.color':'#2a2a2a',
    'grid.linestyle':'--',      'figure.dpi':120,
    'legend.facecolor':'#1a1a1a','legend.edgecolor':'#444',
})

TRACK_WIDTH  = 9.0
N_POINTS     = 800
DEG_TO_M_LAT = 111_320.0
INPUT_SIZE   = 7
D_MODEL      = 64
NHEAD        = 2
NUM_LAYERS   = 3
DIM_FF       = 128
DROPOUT      = 0.1
WINDOW       = 10
SEQ_LEN      = 2*WINDOW+1
BATCH_SIZE   = 256

def deg_to_m_lon(lat_ref):
    return 111_320.0 * np.cos(np.radians(lat_ref))

print(f'✅ Ready  |  Device: {device}  |  PyTorch: {torch.__version__}')

---
## Step 1 — Upload Files

In [ ]:
# ── 1A: Model definition ──────────────────────────────────────────────────
class RacingLineTransformer(nn.Module):
    def __init__(self):
        super().__init__()
        self.input_proj = nn.Linear(INPUT_SIZE, D_MODEL)
        enc = nn.TransformerEncoderLayer(
            d_model=D_MODEL, nhead=NHEAD, dim_feedforward=DIM_FF,
            dropout=DROPOUT, batch_first=True)
        self.transformer = nn.TransformerEncoder(enc, num_layers=NUM_LAYERS)
        self.head = nn.Sequential(
            nn.Linear(D_MODEL,32), nn.ReLU(), nn.Dropout(DROPOUT),
            nn.Linear(32,1),       nn.Sigmoid())
    def forward(self, x):
        x = self.input_proj(x)
        x = self.transformer(x)
        return self.head(x[:, SEQ_LEN//2, :]).squeeze(-1)

total_params = sum(p.numel() for p in RacingLineTransformer().parameters())
print(f'Model parameters : {total_params:,}  ({total_params/1e3:.1f}K)')
print('✅ Model class defined')

In [ ]:
# ── 1B: Upload racing_line_transformer_v8.pth ─────────────────────────────
print('Upload: racing_line_transformer_v8.pth')
up1 = _files.upload()
pth_name = [k for k in up1 if k.endswith('.pth')][0]

model = RacingLineTransformer().to(device)
model.load_state_dict(torch.load(pth_name, map_location=device))
model.eval()

model_size_mb = os.path.getsize(pth_name) / (1024**2)
print(f'✅ Model loaded  |  File size: {model_size_mb:.3f} MB')

In [ ]:
# ── 1C: Upload predictions.json ───────────────────────────────────────────
print('Upload: predictions.json  (from best training run)')
up2 = _files.upload()
pred_name = [k for k in up2 if 'prediction' in k.lower() or k.endswith('.json')][0]

with open(pred_name) as f:
    ALL_PREDICTIONS = json.load(f)

print(f'✅ Predictions loaded  |  Tracks: {len(ALL_PREDICTIONS)}')
print()
print('Available test tracks:')
for i, (t, v) in enumerate(ALL_PREDICTIONS.items()):
    mae = float(np.mean(np.abs(np.array(v['y_true'])-np.array(v['y_pred']))))
    print(f'  {i+1:2d}. {t:<45} MAE={mae:.4f}  ({mae*TRACK_WIDTH*100:.1f} cm)')

In [ ]:
# ── 1D: Upload track CSV ──────────────────────────────────────────────────
# Must be one of the 10 test tracks — needs t_lon, t_lat, alpha columns
# Recommended: Circuit_de_Nevers_Magny-Cours_ground_truth.csv  (best MAE = 32.9 cm)
# Note: any filename works — matching is done by alpha statistics, not filename
print('Upload your track CSV (t_lon, t_lat, alpha columns):')
print('Tip: any filename works — the notebook matches by track data, not filename.')
up3 = _files.upload()
csv_name = list(up3.keys())[0]
df_track = pd.read_csv(io.BytesIO(up3[csv_name]))

missing = [c for c in ['t_lon','t_lat','alpha'] if c not in df_track.columns]
assert not missing, f'Missing columns: {missing}'

# ── Match to predictions.json using THREE methods in order ────────────────
def simplify(s): return re.sub(r'[^a-z0-9]','', s.lower())

csv_alpha_std  = float(df_track['alpha'].std())
csv_alpha_min  = float(df_track['alpha'].min())
csv_alpha_max  = float(df_track['alpha'].max())
csv_lat_mean   = float(df_track['t_lat'].mean())
csv_lon_mean   = float(df_track['t_lon'].mean())

raw_name = (re.sub(r'^\d+[_-]?','', csv_name, flags=re.IGNORECASE)
              .replace('_ground_truth','').replace('ground_truth','')
              .replace('.CSV','').replace('.csv','')
              .replace('_',' ').replace('-',' ').strip())

track_key = None

# Method 1: exact name match
for key in ALL_PREDICTIONS:
    if simplify(key) == simplify(raw_name):
        track_key = key
        print(f'Matched by exact name: "{key}"')
        break

# Method 2: substring name match (only if raw_name is long enough)
if track_key is None and len(simplify(raw_name)) >= 6:
    for key in ALL_PREDICTIONS:
        sk = simplify(key); sr = simplify(raw_name)
        if sk[:8] in sr or sr[:8] in sk:
            track_key = key
            print(f'Matched by name substring: "{key}"')
            break

# Method 3: alpha statistics match (robust to any filename)
# Match by alpha min, max, std — these are unique per track
if track_key is None:
    best_score = float('inf')
    best_key   = None
    for key, v in ALL_PREDICTIONS.items():
        t_std = float(np.std(v['y_true']))
        t_min = float(min(v['y_true']))
        t_max = float(max(v['y_true']))
        score = abs(t_std - csv_alpha_std) + abs(t_min - csv_alpha_min) + abs(t_max - csv_alpha_max)
        if score < best_score:
            best_score = score
            best_key   = key
    if best_score < 0.005:  # tight tolerance — must be very close match
        track_key = best_key
        print(f'Matched by alpha statistics (score={best_score:.6f}): "{best_key}"')
    else:
        print(f'Alpha stats match score {best_score:.4f} is too loose — best candidate: "{best_key}"')

# Method 4: GPS centroid match
if track_key is None:
    best_score = float('inf')
    best_key   = None
    for key, v in ALL_PREDICTIONS.items():
        # We do not have GPS in predictions.json, so use alpha stats with loose tolerance
        t_std = float(np.std(v['y_true']))
        score = abs(t_std - csv_alpha_std)
        if score < best_score:
            best_score = score
            best_key   = key
    track_key = best_key
    print(f'⚠️  Used loose fallback match: "{best_key}" (score={best_score:.4f})')
    print(f'   If this is wrong, re-run and upload the correct CSV.')

TRACK_NAME    = track_key
tr_alpha_pred = np.array(ALL_PREDICTIONS[track_key]['y_pred'])
tr_alpha_true = np.array(ALL_PREDICTIONS[track_key]['y_true'])

# Verify match quality
alpha_match_err = float(np.mean(np.abs(tr_alpha_true - df_track['alpha'].values)))
match_quality   = 'PERFECT ✅' if alpha_match_err < 1e-6 else (f'GOOD ✅ (diff={alpha_match_err:.6f})' if alpha_match_err < 0.01 else f'⚠️  CHECK (diff={alpha_match_err:.4f})')

print(f'\n✅ Track loaded   : {TRACK_NAME}')
print(f'   Points         : {len(df_track)}')
print(f'   Alpha range    : {csv_alpha_min:.3f} → {csv_alpha_max:.3f}')
print(f'   Match quality  : {match_quality}')

---
## Step 2 — Method A: MCP Optimization (Phase 1)
Run the full pipeline fresh and time every sub-step.

In [ ]:
# ── MCP helpers ───────────────────────────────────────────────────────────
def resample_equal_spacing(lon, lat, n, dtm):
    gap = np.sqrt(((lon[-1]-lon[0])*dtm)**2+((lat[-1]-lat[0])*DEG_TO_M_LAT)**2)
    if gap < 1.0: lon,lat = lon[:-1],lat[:-1]
    dx = np.diff(lon)*dtm; dy = np.diff(lat)*DEG_TO_M_LAT
    arc = np.concatenate([[0],np.cumsum(np.sqrt(dx**2+dy**2))])
    total = arc[-1]
    arc_c = np.append(arc, total+np.sqrt(dx[-1]**2+dy[-1]**2))
    fl = interp1d(arc_c, np.append(lon,lon[0])); fa = interp1d(arc_c, np.append(lat,lat[0]))
    s = np.linspace(0,total,n,endpoint=False)
    return fl(s), fa(s), total

def smooth_circular(lon, lat, win=15, poly=3):
    if win%2==0: win+=1
    pad = win*3
    lp  = np.concatenate([lon[-pad:],lon,lon[:pad]])
    lap = np.concatenate([lat[-pad:],lat,lat[:pad]])
    return savgol_filter(lp,win,poly)[pad:-pad], savgol_filter(lap,win,poly)[pad:-pad]

def build_walls(sl, sa, dtm, w=TRACK_WIDTH):
    n = len(sl); nrm = np.zeros((n,2))
    for i in range(n):
        dx=(sl[(i+1)%n]-sl[(i-1)%n])*dtm; dy=(sa[(i+1)%n]-sa[(i-1)%n])*DEG_TO_M_LAT
        nm=np.sqrt(dx**2+dy**2); nm=nm if nm>1e-10 else 1.0
        nrm[i]=np.array([-dy/nm, dx/nm])
    cx,cy=np.mean(sl),np.mean(sa)
    for i in range(n):
        tc=np.array([(cx-sl[i])/dtm,(cy-sa[i])/DEG_TO_M_LAT])
        if np.dot(nrm[i],tc)<0: nrm[i]=-nrm[i]
    nl=nrm[:,0]/dtm; nb=nrm[:,1]/DEG_TO_M_LAT
    mg=np.sqrt(nl**2+nb**2); mg=np.where(mg<1e-15,1,mg); nl/=mg; nb/=mg
    return sl-nl*(w/2)/dtm, sa-nb*(w/2)/DEG_TO_M_LAT, sl+nl*(w/2)/dtm, sa+nb*(w/2)/DEG_TO_M_LAT

def curvature_energy(alpha, il, ia, ol, oa, dtm, lam=0.05):
    n=len(alpha)
    x=(il+alpha*(ol-il))*dtm; y=(ia+alpha*(oa-ia))*DEG_TO_M_LAT
    pi=np.roll(np.arange(n),1); ni=np.roll(np.arange(n),-1)
    ab=np.sqrt((x-x[pi])**2+(y-y[pi])**2)
    bc=np.sqrt((x[ni]-x)**2+(y[ni]-y)**2)
    ac=np.sqrt((x[ni]-x[pi])**2+(y[ni]-y[pi])**2)
    area=np.abs((x-x[pi])*(y[ni]-y[pi])-(x[ni]-x[pi])*(y-y[pi]))/2
    denom=ab*bc*ac
    kappa=np.where(denom>1e-12,(4*area)/denom,0.0)
    return np.sum(kappa**2)+lam*np.sum(np.diff(alpha)**2)

print('✅ MCP helpers defined')

In [ ]:
# ── Run MCP and time every stage ──────────────────────────────────────────
print('Running Phase 1 — MCP Optimization...')
print('(1–3 minutes depending on hardware)')
print('-'*55)

lon_raw = df_track['t_lon'].values.astype(float)
lat_raw = df_track['t_lat'].values.astype(float)
lat_ref = float(np.mean(lat_raw))
DTM_LON = deg_to_m_lon(lat_ref)

mcp_t0 = time.perf_counter()

t0=time.perf_counter(); lon_rs,lat_rs,track_len=resample_equal_spacing(lon_raw,lat_raw,N_POINTS,DTM_LON); t_rs=time.perf_counter()-t0
t0=time.perf_counter(); s_lon,s_lat=smooth_circular(lon_rs,lat_rs); t_sm=time.perf_counter()-t0

area=np.sum(s_lon*np.roll(s_lat,-1)-np.roll(s_lon,-1)*s_lat)/2.0
if area>0: s_lon=s_lon[::-1].copy(); s_lat=s_lat[::-1].copy()

t0=time.perf_counter(); il,ia,ol,oa=build_walls(s_lon,s_lat,DTM_LON); t_wl=time.perf_counter()-t0

n  = len(s_lon)
a0 = 0.5+0.3*np.sin(np.linspace(0,2*np.pi,n))
a0 = np.clip(a0,0.05,0.95)

t0=time.perf_counter()
res=minimize(curvature_energy,a0,args=(il,ia,ol,oa,DTM_LON),method='L-BFGS-B',
             bounds=[(0.05,0.95)]*n,options={'maxiter':2000,'ftol':1e-15,'gtol':1e-8})
mcp_alpha=res.x; t_op=time.perf_counter()-t0

mcp_total = time.perf_counter()-mcp_t0
timing_mcp = {'resample':t_rs,'smooth':t_sm,'walls':t_wl,'optimize':t_op,'total':mcp_total}

print(f'\n✅ MCP complete')
print(f'   Track length  : {track_len:.0f} m  ({track_len/1000:.2f} km)')
print(f'   Optimizer iter: {res.nit}  |  Converged: {res.success}')
print(f'\n   ── Timing Breakdown ──────────────────')
print(f'   Resample       : {t_rs*1000:8.1f} ms')
print(f'   Smooth         : {t_sm*1000:8.1f} ms')
print(f'   Build walls    : {t_wl*1000:8.1f} ms')
print(f'   L-BFGS-B optim : {t_op:8.2f} s')
print(f'   ─────────────────────────────────────')
print(f'   TOTAL MCP      : {mcp_total:8.2f} s  ({mcp_total/60:.2f} min)')

---
## Step 3 — Method B: Transformer Inference (Phase 2)
Load best accuracy from `predictions.json`. Measure live inference time with dummy input.

In [ ]:
# ── Measure inference time with dummy input ────────────────────────────────
# Inference time depends ONLY on architecture + hardware, not on input values.
# We use correctly-shaped random input (800 points × seq_len=21 × 7 features).

print('Measuring Transformer inference time...')
print('(Using dummy input of correct shape: 800 × 21 × 7)')
print('-'*55)

WARMUP_RUNS = 5    # GPU warmup — discard these
TIMING_RUNS = 20   # average over these for stable measurement

dummy_input = torch.randn(N_POINTS, SEQ_LEN, INPUT_SIZE).to(device)

class SimpleDS(Dataset):
    def __init__(self, X): self.X = X
    def __len__(self): return len(self.X)
    def __getitem__(self, i): return self.X[i]

loader = DataLoader(SimpleDS(dummy_input), batch_size=BATCH_SIZE, shuffle=False)

# Warmup
with torch.no_grad():
    for _ in range(WARMUP_RUNS):
        for xb in loader: _ = model(xb)
if device.type == 'cuda': torch.cuda.synchronize()

# Timed runs
run_times = []
for _ in range(TIMING_RUNS):
    t0 = time.perf_counter()
    with torch.no_grad():
        preds_list = []
        for xb in loader:
            preds_list.append(model(xb).cpu().numpy())
    if device.type == 'cuda': torch.cuda.synchronize()
    run_times.append(time.perf_counter()-t0)

tr_time_mean = float(np.mean(run_times))
tr_time_std  = float(np.std(run_times))
speedup      = mcp_total / tr_time_mean

timing_tr = {'inference_mean': tr_time_mean, 'inference_std': tr_time_std,
             'warmup_runs': WARMUP_RUNS, 'timing_runs': TIMING_RUNS}

print(f'\n✅ Inference timing complete')
print(f'   Warmup runs     : {WARMUP_RUNS}')
print(f'   Timing runs     : {TIMING_RUNS}')
print(f'   Mean time       : {tr_time_mean*1000:.2f} ms')
print(f'   Std             : {tr_time_std*1000:.2f} ms')
print(f'   Min/Max         : {min(run_times)*1000:.2f} / {max(run_times)*1000:.2f} ms')
print(f'\n🚀 Transformer is {speedup:.0f}x FASTER than MCP')

In [ ]:
# ── Load accuracy from predictions.json ───────────────────────────────────
alpha_gt   = df_track['alpha'].values          # ground truth from CSV
alpha_pred = tr_alpha_pred                     # from predictions.json best run

# MCP accuracy (vs ground truth — both from same MCP method, small diff expected)
mcp_mae  = float(mean_absolute_error(alpha_gt, mcp_alpha))
mcp_rmse = float(np.sqrt(mean_squared_error(alpha_gt, mcp_alpha)))
mcp_r2   = float(r2_score(alpha_gt, mcp_alpha))
mcp_corr = float(np.corrcoef(alpha_gt, mcp_alpha)[0,1])
mcp_side = float(np.mean((mcp_alpha>0.5)==(alpha_gt>0.5))*100)
mcp_cm   = mcp_mae*TRACK_WIDTH*100

# Transformer accuracy (vs ground truth — from best run)
tr_mae   = float(mean_absolute_error(alpha_gt, alpha_pred))
tr_rmse  = float(np.sqrt(mean_squared_error(alpha_gt, alpha_pred)))
tr_r2    = float(r2_score(alpha_gt, alpha_pred))
tr_corr  = float(np.corrcoef(alpha_gt, alpha_pred)[0,1])
tr_side  = float(np.mean((alpha_pred>0.5)==(alpha_gt>0.5))*100)
tr_cm    = tr_mae*TRACK_WIDTH*100

# Naive baseline (always predict centerline alpha=0.5)
naive_mae = float(mean_absolute_error(alpha_gt, np.full_like(alpha_gt,0.5)))
naive_cm  = naive_mae*TRACK_WIDTH*100

tr_improvement = (naive_mae - tr_mae) / naive_mae * 100

print(f'Accuracy loaded for: {TRACK_NAME}')
print(f'  Naive baseline (centerline) : {naive_cm:.1f} cm')
print(f'  MCP (re-run)                : {mcp_cm:.1f} cm')
print(f'  Transformer (best run)      : {tr_cm:.1f} cm  ({tr_improvement:.1f}% better than baseline)')

---
## Step 4 — Accuracy Comparison Table

In [ ]:
print('='*68)
print(f'  ACCURACY COMPARISON — {TRACK_NAME}')
print('='*68)
print(f'  {"Metric":<28} {"Naive Baseline":>14} {"MCP":>10} {"Transformer":>12}')
print('-'*68)
print(f'  {"MAE (alpha units)":<28} {naive_mae:>14.4f} {mcp_mae:>10.4f} {tr_mae:>12.4f}')
print(f'  {"MAE (cm, 9m track)":<28} {naive_cm:>13.1f}  {mcp_cm:>9.1f}  {tr_cm:>11.1f} ')
print(f'  {"RMSE (alpha units)":<28} {"—":>14} {mcp_rmse:>10.4f} {tr_rmse:>12.4f}')
print(f'  {"R²":<28} {"—":>14} {mcp_r2:>10.4f} {tr_r2:>12.4f}')
print(f'  {"Pearson Correlation":<28} {"—":>14} {mcp_corr:>10.4f} {tr_corr:>12.4f}')
print(f'  {"Correct Side (%)":<28} {"—":>14} {mcp_side:>9.1f}% {tr_side:>11.1f}%')
print('='*68)
print(f'  COMPUTATIONAL COMPARISON')
print('-'*68)
print(f'  {"Computation time":<28} {"—":>14} {mcp_total:>9.2f}s {tr_time_mean*1000:>10.1f}ms')
print(f'  {"Speed advantage":<28} {"—":>14} {"baseline":>10} {"{:.0f}x faster".format(speedup):>12}')
print(f'  {"Model file size":<28} {"—":>14} {"N/A":>10} {"{:.3f} MB".format(model_size_mb):>12}')
print(f'  {"Parameters":<28} {"—":>14} {"N/A":>10} {"{:,}".format(total_params):>12}')
print(f'  {"Requires scipy":<28} {"—":>14} {"Yes":>10} {"No":>12}')
print(f'  {"Requires per-track optim":<28} {"—":>14} {"Yes":>10} {"No":>12}')
print(f'  {"Portable to embedded HW":<28} {"—":>14} {"Difficult":>10} {"Yes":>12}')
print('='*68)

# All 10 test tracks summary from predictions.json
print(f'\n  ALL 10 TEST TRACKS (Transformer — Best Run)')
print('-'*68)
all_maes = []
for t, v in ALL_PREDICTIONS.items():
    m = float(np.mean(np.abs(np.array(v['y_true'])-np.array(v['y_pred']))))
    all_maes.append(m)
    grade = 'Excellent' if m*TRACK_WIDTH*100 < 50 else ('Acceptable' if m*TRACK_WIDTH*100 < 100 else 'Needs work')
    print(f'  {t:<45} {m*TRACK_WIDTH*100:>6.1f} cm  {grade}')
print('-'*68)
print(f'  Mean across all 10 tracks: {np.mean(all_maes)*TRACK_WIDTH*100:.1f} cm')
print(f'  Naive baseline:            {naive_cm:.1f} cm')
print(f'  Overall improvement:       {(naive_mae-np.mean(all_maes))/naive_mae*100:.1f}%')
print('='*68)

---
## Step 5 — Visualisations

In [ ]:
# ── Plot 1: Alpha profile comparison ──────────────────────────────────────
idx = np.arange(N_POINTS)
mcp_err_cm = np.abs(mcp_alpha - alpha_gt) * TRACK_WIDTH * 100
tr_err_cm  = np.abs(alpha_pred - alpha_gt) * TRACK_WIDTH * 100

fig, axes = plt.subplots(3,1,figsize=(16,12),sharex=True)
fig.suptitle(f'{TRACK_NAME}\nAlpha Profile — Ground Truth vs MCP vs Transformer',
             fontsize=13, fontweight='bold', y=1.01)

axes[0].plot(idx, alpha_gt,    color='#90CAF9',lw=1.2,label='Ground Truth')
axes[0].plot(idx, mcp_alpha,   color='#A5D6A7',lw=1.0,ls='--',alpha=0.9,
             label=f'MCP (re-run)   MAE={mcp_cm:.1f} cm')
axes[0].plot(idx, alpha_pred,  color='#EF9A9A',lw=1.0,ls='-.',alpha=0.9,
             label=f'Transformer    MAE={tr_cm:.1f} cm')
axes[0].axhline(0.5,color='#555',lw=0.7,ls=':',label='Centreline α=0.5')
axes[0].set_ylabel('Alpha'); axes[0].set_ylim(-0.05,1.05)
axes[0].legend(fontsize=9,loc='upper right'); axes[0].grid(True,alpha=0.3)
axes[0].set_title('Lateral Offset α Along Track',fontsize=10)

axes[1].fill_between(idx,mcp_err_cm,alpha=0.55,color='#A5D6A7')
axes[1].axhline(mcp_err_cm.mean(),color='#A5D6A7',lw=1.5,ls='--',
                label=f'Mean = {mcp_err_cm.mean():.1f} cm')
axes[1].set_ylabel('Error (cm)'); axes[1].legend(fontsize=9)
axes[1].grid(True,alpha=0.3); axes[1].set_title('MCP Error vs Ground Truth',fontsize=10)

axes[2].fill_between(idx,tr_err_cm,alpha=0.55,color='#EF9A9A')
axes[2].axhline(tr_err_cm.mean(),color='#EF9A9A',lw=1.5,ls='--',
                label=f'Mean = {tr_err_cm.mean():.1f} cm')
axes[2].set_xlabel('Track Point Index'); axes[2].set_ylabel('Error (cm)')
axes[2].legend(fontsize=9); axes[2].grid(True,alpha=0.3)
axes[2].set_title('Transformer Error vs Ground Truth',fontsize=10)

plt.tight_layout()
plt.savefig('plot1_alpha_profile.png',bbox_inches='tight',dpi=130)
plt.show(); print('Saved: plot1_alpha_profile.png')

In [ ]:
# ── Plot 2: Racing line on track map ──────────────────────────────────────
lon_v = df_track['t_lon'].values; lat_v = df_track['t_lat'].values
dtm_v = deg_to_m_lon(float(np.mean(lat_v)))
dx_v  = np.gradient(lon_v); dy_v = np.gradient(lat_v)
spd_v = np.sqrt((dx_v*dtm_v)**2+(dy_v*DEG_TO_M_LAT)**2)+1e-9
px_v  = -dy_v*DEG_TO_M_LAT/spd_v/dtm_v
py_v  =  dx_v*dtm_v/spd_v/DEG_TO_M_LAT
hw_l  = (TRACK_WIDTH/2)/dtm_v; hw_a = (TRACK_WIDTH/2)/DEG_TO_M_LAT
in_l  = lon_v-px_v*hw_l; in_a = lat_v-py_v*hw_a
ou_l  = lon_v+px_v*hw_l; ou_a = lat_v+py_v*hw_a

def alpha2lonlat(a):
    return in_l+a*(ou_l-in_l), in_a+a*(ou_a-in_a)

gt_lo,gt_la   = alpha2lonlat(alpha_gt)
mcp_lo,mcp_la = alpha2lonlat(mcp_alpha)
tr_lo,tr_la   = alpha2lonlat(alpha_pred)

fig,axes = plt.subplots(1,3,figsize=(21,7))
fig.suptitle(f'{TRACK_NAME} — Racing Line on Track Map',fontsize=13,fontweight='bold')

configs = [
    ('Ground Truth',    gt_lo,  gt_la,  np.zeros(N_POINTS), '#90CAF9'),
    ('MCP (re-run)',    mcp_lo, mcp_la, mcp_err_cm,         '#A5D6A7'),
    ('Transformer',     tr_lo,  tr_la,  tr_err_cm,          '#EF9A9A'),
]
max_err = max(mcp_err_cm.max(), tr_err_cm.max())

for ax,(title,rlo,rla,err,col) in zip(axes,configs):
    ax.plot(np.append(in_l,in_l[0]),np.append(in_a,in_a[0]),'-',color='#fff',lw=0.9)
    ax.plot(np.append(ou_l,ou_l[0]),np.append(ou_a,ou_a[0]),'-',color='#fff',lw=0.9)
    ax.plot(np.append(lon_v,lon_v[0]),np.append(lat_v,lat_v[0]),'--',
            color='#546e7a',lw=0.5,alpha=0.4)
    pts  = np.array([rlo,rla]).T.reshape(-1,1,2)
    segs = np.concatenate([pts[:-1],pts[1:]],axis=1)
    if err.max()>0:
        lc=LineCollection(segs,cmap='RdYlGn_r',norm=plt.Normalize(0,max_err))
        lc.set_array(err); lc.set_linewidth(2.5); ax.add_collection(lc)
        plt.colorbar(lc,ax=ax,label='Error (cm)',shrink=0.7)
    else:
        ax.plot(rlo,rla,'-',color=col,lw=2.0)
    ax.set_title(title,fontsize=11,fontweight='bold')
    ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')
    ax.set_aspect('equal'); ax.grid(True,alpha=0.2); ax.tick_params(labelsize=7)

plt.tight_layout()
plt.savefig('plot2_track_map.png',bbox_inches='tight',dpi=130)
plt.show(); print('Saved: plot2_track_map.png')

In [ ]:
# ── Plot 3: Summary bar charts ─────────────────────────────────────────────
fig,axes = plt.subplots(1,3,figsize=(16,5))
fig.suptitle('MCP vs Transformer — Summary',fontsize=13,fontweight='bold')

methods = ['Naive\nBaseline','MCP\n(Phase 1)','Transformer\n(Phase 2)']
clrs    = ['#78909C','#A5D6A7','#EF9A9A']

# MAE
vals = [naive_cm, mcp_cm, tr_cm]
bars = axes[0].bar(methods,vals,color=clrs,edgecolor='#111',width=0.5)
for b,v in zip(bars,vals):
    axes[0].text(b.get_x()+b.get_width()/2, v+0.8, f'{v:.1f}cm',
                 ha='center',fontsize=11,fontweight='bold')
axes[0].set_title(f'MAE — {TRACK_NAME[:25]}',fontsize=10)
axes[0].set_ylabel('Error (cm)'); axes[0].grid(True,axis='y',alpha=0.3)

# Correct side %
vals2 = [50.0, mcp_side, tr_side]
bars2 = axes[1].bar(methods,vals2,color=clrs,edgecolor='#111',width=0.5)
for b,v in zip(bars2,vals2):
    axes[1].text(b.get_x()+b.get_width()/2, v-4, f'{v:.1f}%',
                 ha='center',fontsize=11,fontweight='bold',color='#111')
axes[1].set_title('Correct Side Prediction',fontsize=10)
axes[1].set_ylabel('% of points'); axes[1].set_ylim(0,108)
axes[1].grid(True,axis='y',alpha=0.3)

# Computation time (log)
vals3 = [None, mcp_total*1000, tr_time_mean*1000]
axes[2].bar(['MCP\n(Phase 1)','Transformer\n(Phase 2)'],
            [mcp_total*1000, tr_time_mean*1000],
            color=['#A5D6A7','#EF9A9A'],edgecolor='#111',width=0.5)
for x,(v,lbl) in enumerate([(mcp_total*1000,f'{mcp_total:.1f}s'),
                              (tr_time_mean*1000,f'{tr_time_mean*1000:.1f}ms')]):
    axes[2].text(x, v*1.15, lbl, ha='center',fontsize=11,fontweight='bold')
axes[2].set_yscale('log')
axes[2].set_title(f'Computation Time (log scale)\n{speedup:.0f}× speed advantage',fontsize=10)
axes[2].set_ylabel('Time (ms, log)'); axes[2].grid(True,axis='y',alpha=0.3)

plt.tight_layout()
plt.savefig('plot3_summary_bars.png',bbox_inches='tight',dpi=130)
plt.show(); print('Saved: plot3_summary_bars.png')

In [ ]:
# ── Plot 4: All 10 test tracks — Transformer MAE bar ─────────────────────
import matplotlib.patches as mpatches

track_short = [t.replace(' Circuit','').replace(' International','').replace(' Motor Speedway','')
               .replace(' Corniche','')[:22] for t in ALL_PREDICTIONS.keys()]
maes_cm_all = [float(np.mean(np.abs(np.array(v['y_true'])-np.array(v['y_pred']))))*TRACK_WIDTH*100
               for v in ALL_PREDICTIONS.values()]

colors_all = ['#66BB6A' if m<50 else ('#FFA726' if m<100 else '#EF5350') for m in maes_cm_all]

fig,ax = plt.subplots(figsize=(14,5))
bars = ax.bar(range(len(maes_cm_all)), maes_cm_all, color=colors_all, edgecolor='#111')
ax.axhline(np.mean(maes_cm_all), color='#fff', lw=1.5, ls='--',
           label=f'Mean = {np.mean(maes_cm_all):.1f} cm')
ax.axhline(naive_cm, color='#78909C', lw=1.2, ls=':',
           label=f'Naive baseline = {naive_cm:.1f} cm')
for i,(b,v) in enumerate(zip(bars,maes_cm_all)):
    ax.text(i, v+2, f'{v:.0f}', ha='center', fontsize=8, fontweight='bold')
ax.set_xticks(range(len(maes_cm_all)))
ax.set_xticklabels(track_short, rotation=35, ha='right', fontsize=9)
ax.set_ylabel('MAE (cm)'); ax.set_title('Transformer — MAE on All 10 Test Tracks (Best Run)',
                                          fontsize=12, fontweight='bold')
green  = mpatches.Patch(color='#66BB6A', label='Excellent (<50 cm)')
orange = mpatches.Patch(color='#FFA726', label='Acceptable (50–100 cm)')
red_p  = mpatches.Patch(color='#EF5350', label='Needs work (>100 cm)')
ax.legend(handles=[green,orange,red_p]+ax.get_legend_handles_labels()[0][-2:], fontsize=9)
ax.grid(True,axis='y',alpha=0.3)
plt.tight_layout()
plt.savefig('plot4_all_tracks.png',bbox_inches='tight',dpi=130)
plt.show(); print('Saved: plot4_all_tracks.png')

---
## Step 6 — Engineering Trade-off Summary

In [ ]:
print('='*70)
print('  ENGINEERING TRADE-OFF SUMMARY')
print('  Shell Eco-Marathon Racing Line Generation')
print('='*70)
print(f'  {"Criterion":<32} {"MCP (Phase 1)":<22} {"Transformer (Phase 2)"}')
print('-'*70)
print(f'  {"Alpha MAE (this track)":<32} {f"{mcp_cm:.1f} cm":<22} {f"{tr_cm:.1f} cm"}')
print(f'  {"Alpha MAE (10 track mean)":<32} {"N/A (ground truth)":<22} {f"{np.mean(all_maes)*TRACK_WIDTH*100:.1f} cm"}')
print(f'  {"vs Naive Baseline":<32} {"Ground truth":<22} {f"{(naive_mae-np.mean(all_maes))/naive_mae*100:.1f}% better"}')
print(f'  {"Pearson Correlation":<32} {f"{mcp_corr:.4f}":<22} {f"{tr_corr:.4f}"}')
print(f'  {"Correct side prediction":<32} {f"{mcp_side:.1f}%":<22} {f"{tr_side:.1f}%"}')
print('-'*70)
print(f'  {"Computation time":<32} {f"{mcp_total:.1f}s ({mcp_total/60:.1f} min)":<22} {f"{tr_time_mean*1000:.1f}ms"}')
print(f'  {"Speed advantage":<32} {"baseline":<22} {f"{speedup:.0f}× faster"}')
print(f'  {"Model size":<32} {"N/A":<22} {f"{model_size_mb:.3f} MB"}')
print(f'  {"Parameters":<32} {"N/A":<22} {f"{total_params:,}"}')
print(f'  {"Requires scipy optimizer":<32} {"Yes":<22} {"No"}')
print(f'  {"Per-track computation":<32} {"Yes (~2 min/track)":<22} {"No (instant)"}')
print(f'  {"Works on new tracks":<32} {"Yes (re-optimize)":<22} {"Yes (instant inference)"}')
print(f'  {"Portable to embedded HW":<32} {"Difficult":<22} {"Yes (PyTorch/ONNX)"}')
print(f'  {"Internet required":<32} {"No":<22} {"No"}')
print('='*70)
print()
print('  KEY ENGINEERING INSIGHT:')
print(f'  MCP is the accuracy gold standard — it IS the ground truth.')
print(f'  The Transformer approximates MCP quality in {tr_time_mean*1000:.0f}ms instead of {mcp_total:.0f}s,')
print(f'  making it {speedup:.0f}× faster and suitable for onboard embedded deployment')
print(f'  without scipy, iterative solvers, or per-track setup time.')
print(f'  On 9m-wide SEM tracks, {tr_cm:.1f}cm mean error is within half a tyre width.')
print('='*70)

---
## Step 7 — Save Metrics & Download

In [ ]:
import zipfile

# Save metrics CSV
metrics = pd.DataFrame({
    'Method'            : ['Naive Baseline','MCP (Phase 1)','Transformer (Phase 2)'],
    'MAE_alpha'         : [round(naive_mae,6), round(mcp_mae,6),  round(tr_mae,6)],
    'MAE_cm'            : [round(naive_cm,2),  round(mcp_cm,2),   round(tr_cm,2)],
    'RMSE_alpha'        : ['—',                round(mcp_rmse,6), round(tr_rmse,6)],
    'R2'                : ['—',                round(mcp_r2,4),   round(tr_r2,4)],
    'Pearson_corr'      : ['—',                round(mcp_corr,4), round(tr_corr,4)],
    'Correct_side_pct'  : ['—',                round(mcp_side,2), round(tr_side,2)],
    'Time_seconds'      : ['—',                round(mcp_total,3),round(tr_time_mean,5)],
    'Speedup_x'         : ['—',                1.0,               round(speedup,1)],
    'Model_size_MB'     : ['—',                'N/A',             round(model_size_mb,3)],
    'Parameters'        : ['—',                'N/A',             total_params],
})
metrics.to_csv('comparison_metrics.csv',index=False)

# Bundle
output_files = [
    'comparison_metrics.csv',
    'plot1_alpha_profile.png',
    'plot2_track_map.png',
    'plot3_summary_bars.png',
    'plot4_all_tracks.png',
]
with zipfile.ZipFile('comparative_analysis_outputs.zip','w') as zf:
    for f in output_files:
        if os.path.exists(f): zf.write(f); print(f'  Added: {f}')

_files.download('comparative_analysis_outputs.zip')
print('\n✅ All outputs downloaded.')
print('   plot1 — Alpha profile (3 panels: overlay, MCP error, Transformer error)')
print('   plot2 — Racing line on track map (3 panels coloured by error magnitude)')
print('   plot3 — Summary bars (MAE, correct side %, computation time)')
print('   plot4 — All 10 test tracks MAE (green/orange/red grading)')
print('   comparison_metrics.csv — full numerical comparison')